# SIOP 2026 - Agentic AI Master Tutorial

APIs: OpenAI key, O*NET key (optional, mock fallback)

Andrew Samo, PhD

---

 We just saw this workflow as boxes and arrows in n8n. Now we're going to show you what's underneath. An agent is really just a loop, two tools, some state, and quality gates.


### Environment Setup

In [ ]:
!pip install -q openai pandas requests

In [ ]:
import os
import json
import requests
import pandas as pd
from getpass import getpass
from openai import OpenAI

In [ ]:
# Set API keys
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)
print("OAI Key Set")

onet_key = userdata.get("ONET_API_KEY")
os.environ["ONET_API_KEY"] = onet_key if onet_key else ""

OAI Key Set
ONET Key Set


In [ ]:
# Is the live O*NET API available? Controls agent search strategy.
ONET_AVAILABLE = bool(os.getenv("ONET_API_KEY", "").strip())
print(f"O*NET live API: {'available' if ONET_AVAILABLE else '✗ using mock data'}")

O*NET live API: available


In [ ]:
# Set models

# Notce we're using two different models.
# The agent, which does the reasoning and searching, runs on gpt-4.1, a cheap, effective agent model
# The evaluators run on 4.1-mini, a smaller, cheaper model that runs a checklist (criteria)

AGENT_MODEL = "gpt-4.1" # good agent model
EVAL_MODEL = "gpt-4.1-mini" # and smaller models are good helpers
WEB_MODEL = "gpt-4.1"

In [ ]:
# JSON Helper

# The helper is the only interface between our code and the LLM,
# so every call in this notebook goes through this fx
# We're using JSON so the model *always* returns structured data
# we can then parse over feeeform text, which is nice
# because the rest of our code/ evals need to make decisions
# based on what the model is saying

def llm_json(prompt, model=AGENT_MODEL, system="You are a careful I-O psychology assistant."):
    """Call the LLM and get back parsed JSON."""
    response = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system + " Return only valid JSON. No markdown."},
            {"role": "user",   "content": prompt},
        ],
    )
    return json.loads(response.choices[0].message.content)

## Input: Job Description

For this demo we are using a pre-loaded "Customer Success Manager" JD. In practicem you'd paste in whatever role you're working with.

- We can also use a crowd-sourced JD here

In [ ]:
# Set Input/ Job Description

job_description = """
Customer Success Manager responsible for onboarding clients, managing relationships,
resolving issues, analyzing account data, and coordinating with internal teams.
Requires strong communication, problem solving, customer orientation, data interpretation,
and project management.
"""

## Set up the agent memory (state)

In [ ]:
# Initialize and set agent state
# This is, basically, the agents entire memory - it's a python dictionary
# so no fancy vector DBs, just a dict with keys for evidence, search history,
# and status flags -- super basic memory
# as you scale up in production, you'd want to expand memory with more space (DBs)
# or memory management/ compression

state = {
    "job_description": job_description,
    "evidence": [],
    "search_history": [],
    "trace": [],
    "evidence_sufficient": False,
    "guide": None,
    "guide_review": None,
    "guide_approved": False,
    "revision_count": 0,
    "done": False,
    "step": 0,
}

print("Agent state initialized ✓")
print(f"  Keys: {list(state.keys())}")

Agent state initialized ✓
  Keys: ['job_description', 'evidence', 'search_history', 'trace', 'evidence_sufficient', 'guide', 'guide_review', 'guide_approved', 'revision_count', 'done', 'step']


## Set up tools (web search, O*NET search)

We're giving the agent exactly two tools here - web search and ONET lookup. These are the things the LLM cannot do on its own (i.e., external capabilities; e.g., search online, query a DB) but everything else, like reading the job description, extracting competencies, writing questions, is just an LLM doing LLM stuff (internal capability).

In [ ]:
# define tool #1 - Web Search

def web_search(query):
    """Search the web and return a text summary as evidence."""
    try:
        response = client.responses.create(
            model=WEB_MODEL,
            tools=[{"type": "web_search"}],
            input=(
                f"Search the web for occupationally relevant information about: {query}\n\n"
                "Summarize what would be useful for creating job-relevant structured "
                "interview questions. Keep it concise."
            ),
        )
        summary = response.output_text.strip()
    except Exception as e:
        summary = f"Web search unavailable ({e}). Proceeding with existing evidence."

    return {
        "source": "web_search",
        "query": query,
        "summary": summary,
    }

print("Tool: web_search active")


In [ ]:
# define tool #2 - O*NET Search

# Uses the live O*NET Web Services API if a key is provided.
# Falls back to realistic mock data so the demo never fails on stage.

def search_onet(query, max_results=3):
    """Search O*NET for occupations matching a keyword."""
    api_key = os.getenv("ONET_API_KEY", "").strip()

    # ── Mock fallback ────────────────────────────────────────────────────
    # Deliberately thin — gives occupation matches but not enough detail
    # for the evaluator to approve immediately. Forces web search follow-up.
    mock_data = {
        "source": "O*NET (mock)",
        "query": query,
        "occupations": [
            {"title": "Customer Service Representatives", "code": "43-4051.00"},
            {"title": "Sales Managers", "code": "11-2022.00"},
        ],
        "summary": (
            "Closest O*NET matches: Customer Service Representatives (43-4051.00) "
            "and Sales Managers (11-2022.00). Further evidence needed on specific "
            "tasks, competencies, and KSAs for this role."
        ),
    }

    if not api_key:
        return mock_data

    # ── Live O*NET Web Services v2 call ──────────────────────────────────
    # Auth: X-API-Key header (per O*NET v2 API)
    try:
        url = "https://api-v2.onetcenter.org/online/search"
        headers = {
            "X-API-Key": api_key,
            "Accept": "application/json"
        }
        params = {
            "keyword": query,
            "end": max_results
        }

        r = requests.get(url, headers=headers, params=params, timeout=15)
        r.raise_for_status()
        data = r.json()

        occupations = data.get("occupation", [])[:max_results]

        return {
            "source": "O*NET",
            "query": query,
            "summary": f"Top O*NET matches for '{query}': "
                       + ", ".join(o.get("title", "?") for o in occupations),
            "items": [
                {
                    "title": o.get("title"),
                    "code": o.get("code"),
                    "href": o.get("href")
                }
                for o in occupations
            ],
        }

    except Exception as e:
        print(f"  ⚠ O*NET API error ({e}), using mock data.")
        mock_data["source"] = "O*NET (mock — API fallback)"
        return mock_data

print("Tool: search_onet active")


Tool: search_onet active


In [ ]:
# check that the O*NET API is working (we want to see code: 200)

api_key = os.getenv("ONET_API_KEY", "").strip()

r = requests.get(
    "https://api-v2.onetcenter.org/about/",
    headers={"X-API-Key": api_key, "Accept": "application/json"},
    timeout=15
)

print(r.status_code)
print(r.text[:500])

200
{"taxonomy":{"url":"https://www.onetcenter.org/taxonomy.html","name":"O*NET-SOC 2019"},"database":{"url":"https://www.onetcenter.org/database.html","name":"O*NET 30.2"},"api_version":"2.0.0"}


## Set agent and evaluator instructions
- tool counter
- search agent prompt
- evaluator 1: search evidence
- interview builder prompt
- evaluator 2: interview evidence

In [ ]:
def count_tool_uses(state, tool_name):
    return sum(1 for a in state["search_history"] if a.get("action") == tool_name)

# This is the core "agent" prompt, where the agent is given its task, instructions
# tools, and actually decides what to search for. We're telling it what tools it has
# access too and what it should be doing to reach a goal

def decide_next_action(state):
    """Ask the agent LLM which tool to use next for evidence gathering."""

    onet_count = count_tool_uses(state, "search_onet")
    web_count = count_tool_uses(state, "web_search")

    prompt = f"""You are the main agent in an I-O psychology workflow.

Goal: Gather enough job-relevant evidence to create a structured behavioral
interview guide.

Available tools:
1. "search_onet" — use for occupational titles, tasks, skills, abilities, work activities.
2. "web_search" — use for broader job-market, role, or domain context.

O*NET live API available: {ONET_AVAILABLE}
If O*NET is not live, use search_onet at most once for baseline occupation
matching, then prefer web_search for richer evidence.

Tool use so far:
search_onet={onet_count},
web_search={web_count}

Search policy:
- Build a THOROUGH evidence file before drafting. Aim for 8-10 evidence items.
- Alternate between search_onet and web_search to get diverse evidence.
- Use O*NET for: occupation matching, task statements, skills, abilities,
  knowledge areas, work activities, work context.
- Use web search for: role responsibilities, industry expectations, competency
  models, common challenges, performance metrics, team dynamics.
- Each search should target a DIFFERENT aspect of the role.
- Do not repeat a prior search query.
- Keep the query short and specific (3-8 words).

Job description:
{state["job_description"]}

Evidence gathered so far:
{json.dumps(state["evidence"], indent=2)}

Search history (do NOT repeat these):
{json.dumps(state["search_history"], indent=2)}

Return JSON:
{{"action": "search_onet" or "web_search", "query": "...", "reason": "..."}}"""

    return llm_json(prompt, model=AGENT_MODEL)

In [ ]:
# Process Evaluator (LLM-as-Judge)

# this is a seperate LLM call with a completely different job -it's judging whether
# the evidence is sufficient. The criteria involve ONET matches, task details, KSA
# specificty, etc. which are IO Psych style criteria.

# For most IO's involved in agent building, IMO this is where the value is!!!

def evaluate_evidence(state):
    """Mini-evaluator: is the evidence sufficient to draft interview questions?"""
    prompt = f"""You are a mini-evaluator for an agentic I-O psychology workflow.

Decide whether the evidence file is sufficient to create a defensible first
draft of structured behavioral interview questions.

Job description:
{state["job_description"]}

Evidence file:
{json.dumps(state["evidence"], indent=2)}

Criteria — evidence is sufficient when ALL of the following are met:
1. At least one plausible O*NET occupation match with task or KSA detail
2. Multiple job-relevant tasks identified from different sources
3. Multiple competencies, skills, or KSAs identified with behavioral specificity
4. Broader role context (e.g., challenges, performance expectations, team dynamics)
5. Enough detail to support 5 behavioral interview questions with scoring anchors

IMPORTANT: A thorough evidence file typically requires 8-10 evidence items
from a mix of O*NET and web sources. If fewer than 6 evidence items exist,
return sufficient: false. Quality matters — occupation titles alone without
task or competency detail do not count as substantive evidence.

Err on the side of "sufficient" once multiple substantive sources are gathered —
a good first draft with human review is better than endlessly gathering evidence.

Return JSON:
{{"sufficient": true or false, "reason": "...", "missing": ["...", "..."]}}"""

    return llm_json(prompt, model=EVAL_MODEL)


In [ ]:
# define action (function) to generate interview guide

def generate_interview_guide(state):
    """The LLM produces the work product using the evidence file."""
    prompt = f"""You are an I-O psychologist creating a structured behavioral interview guide.

Job description:
{state["job_description"]}

Evidence file:
{json.dumps(state["evidence"], indent=2)}

Create exactly 5 behavioral interview questions.

Requirements:
- Each question must target a job-relevant competency.
- Each question must be behavioral (past-tense, situation-based), not hypothetical.
- Each question must cite evidence from the JD, O*NET data, or web evidence.
- Include simple 1-3-5 scoring anchors (1 = poor, 3 = acceptable, 5 = strong).
- Avoid protected-class, medical, family-status, disability, age, or other
  legally risky content.

Return JSON:
{{"guide": [
  {{"competency": "...", "question": "...", "evidence": "...",
    "anchor_1": "...", "anchor_3": "...", "anchor_5": "..."}}
]}}"""

    return llm_json(prompt, model=AGENT_MODEL)["guide"]

In [ ]:
# define evaluator 2 - for interview guide

def evaluate_interview_guide(guide, state):
    """Mini-evaluator: is the interview guide good enough for human review?"""
    prompt = f"""You are a mini-evaluator reviewing a structured interview guide.

Job description:
{state["job_description"]}

Evidence file:
{json.dumps(state["evidence"], indent=2)}

Interview guide:
{json.dumps(guide, indent=2)}

Evaluate against this rubric:
1. Questions are behavioral (past-tense, situation-based).
2. Questions map to job-relevant competencies.
3. Questions cite evidence from the JD or evidence file.
4. Scoring anchors meaningfully distinguish low / medium / high responses.
5. No obvious legal, ethical, or fairness red flags.

Return JSON:
{{"approved": true or false, "verdict": "...",
  "issues": [{{"question_number": 1, "issue": "...", "suggested_fix": "..."}}]}}"""

    return llm_json(prompt, model=EVAL_MODEL)

In [ ]:
# revise interview guide (if eval flags issues)

def revise_interview_guide(state):
    """Revise the guide based on evaluator feedback."""
    prompt = f"""Revise the structured interview guide based on evaluator feedback.

Job description:
{state["job_description"]}

Evidence file:
{json.dumps(state["evidence"], indent=2)}

Current guide:
{json.dumps(state["guide"], indent=2)}

Evaluator feedback:
{json.dumps(state["guide_review"], indent=2)}

Fix only the flagged issues. Keep questions that passed unchanged.

Return JSON:
{{"guide": [
  {{"competency": "...", "question": "...", "evidence": "...",
    "anchor_1": "...", "anchor_3": "...", "anchor_5": "..."}}
]}}"""

    return llm_json(prompt, model=AGENT_MODEL)["guide"]


In [ ]:
# helper - log short summary so far
def short_summary(result, max_chars=160):
    text = result.get("summary", "")
    return text[:max_chars] + ("..." if len(text) > max_chars else "")

# Agent Loop 🔁

This is the main agent loop, where everything else can be thought of as the setup.

We start it up and the agent will iteratively consider it's goal, use tools to gather information, send it over to the evalautors until they pass the work and it is sent to the next phase where they are designing interview questions.

It's also good practice to print out the steps, tool calls, and reasoning for an auditable trace.


In [ ]:
max_steps = 15

print("=" * 65)
print("  🔁 AGENT LOOP")
print("=" * 65)

while not state["done"] and state["step"] < max_steps:
    state["step"] += 1
    step = state["step"]

    # ── Phase 1: Evidence gathering ──────────────────────────────────────
    if not state["evidence_sufficient"]:

        action = decide_next_action(state)
        tool = action.get("action", "web_search")
        query = action.get("query", "")
        reason = action.get("reason", "")

        if tool == "search_onet":
            result = search_onet(query)
        else:
            result = web_search(query)

        state["evidence"].append(result)
        state["search_history"].append(action)

        review = evaluate_evidence(state)
        state["evidence_sufficient"] = review.get("sufficient", False)

        state["trace"].append({
            "step": step,
            "phase": "Evidence Gathering",
            "action": tool,
            "detail": query,
            "evaluator": "SUFFICIENT" if review["sufficient"] else "CONTINUE",
            "reason": review.get("reason", ""),
        })

        print(f"\n  Step {step} · Evidence Gathering")
        print(f"    Tool:      {tool}")
        print(f"    Query:     {query}")
        print(f"    Why:       {reason}")
        print(f"    Result:    {short_summary(result)}")
        ev = "✅ SUFFICIENT" if review["sufficient"] else "⏳ CONTINUE"
        print(f"    Evaluator: {ev}")
        print(f"    Reason:    {review.get('reason', '')}")

    # ── Phase 2: Draft the interview guide ───────────────────────────────
    elif state["guide"] is None:

        guide = generate_interview_guide(state)
        state["guide"] = guide

        review = evaluate_interview_guide(guide, state)
        state["guide_review"] = review
        state["guide_approved"] = review.get("approved", False)

        state["trace"].append({
            "step": step,
            "phase": "Draft Guide",
            "action": "generate_interview_guide",
            "detail": f"Generated {len(guide)} questions",
            "evaluator": "APPROVED" if review["approved"] else "REVISE",
            "reason": review.get("verdict", ""),
        })

        ev = "✅ APPROVED" if review["approved"] else "🔄 REVISE"
        print(f"\n  Step {step} · Draft Interview Guide")
        print(f"    Generated: {len(guide)} interview questions")
        print(f"    Evaluator: {ev}")
        print(f"    Verdict:   {review.get('verdict', '')}")

        if review.get("issues"):
            for issue in review["issues"]:
                qn = issue.get("question_number", "?")
                print(f"    ⚠ Q{qn}: {issue.get('issue', '')}")

    # ── Phase 3: Revise if evaluator flagged issues ──────────────────────
    elif not state["guide_approved"]:

        # Guardrail: max 2 revisions, then escalate to human review
        if state["revision_count"] >= 2:
            state["done"] = True
            state["trace"].append({
                "step": step,
                "phase": "Escalate to Human",
                "action": "max_revisions_reached",
                "detail": "Guide not yet approved after 2 revisions. Escalating to human review.",
                "evaluator": "HUMAN REVIEW",
                "reason": "Revision limit reached — a human should take it from here.",
            })
            print(f"\n  Step {step} · ⚠ Max Revisions Reached")
            print("    Guide sent to human review after 2 revision attempts.")
            continue

        revised = revise_interview_guide(state)
        state["guide"] = revised
        state["revision_count"] += 1

        review = evaluate_interview_guide(revised, state)
        state["guide_review"] = review
        state["guide_approved"] = review.get("approved", False)

        state["trace"].append({
            "step": step,
            "phase": "Revise Guide",
            "action": "revise_interview_guide",
            "detail": f"Revised {len(revised)} questions",
            "evaluator": "APPROVED" if review["approved"] else "REVISE",
            "reason": review.get("verdict", ""),
        })

        ev = "✅ APPROVED" if review["approved"] else "🔄 REVISE"
        print(f"\n  Step {step} · Revise Interview Guide")
        print(f"    Revised:   {len(revised)} questions")
        print(f"    Evaluator: {ev}")
        print(f"    Verdict:   {review.get('verdict', '')}")

    # ── Phase 4: Done ────────────────────────────────────────────────────
    else:
        state["done"] = True
        state["trace"].append({
            "step": step,
            "phase": "Done",
            "action": "finalize",
            "detail": "Guide approved. Ready for human review.",
            "evaluator": "DONE",
            "reason": "All quality gates passed.",
        })
        print(f"\n  Step {step} · ✅ Done")
        print("    Guide approved. Ready for human review.")

print("\n" + "=" * 65)
print(f"  Loop complete in {state['step']} steps.")
print("=" * 65)


  🔁 AGENT LOOP

  Loop complete in 9 steps.


In [ ]:
# Audit Trail for # Observability

print("\n📋 Agent Trace\n")
trace_df = pd.DataFrame(state["trace"])
trace_df


📋 Agent Trace



,step,phase,action,detail,evaluator,reason
0,1,Evidence Gathering,search_onet,Customer Success Manager tasks,CONTINUE,The evidence file contains only one source wit...
1,2,Evidence Gathering,search_onet,Customer Success Manager KSAs,CONTINUE,The evidence file contains only two sources wh...
2,3,Evidence Gathering,web_search,Customer Success Manager key competencies,CONTINUE,The evidence file includes plausible O*NET occ...
3,4,Evidence Gathering,web_search,Customer Success Manager common challenges,SUFFICIENT,The evidence file includes plausible O*NET occ...
4,5,Draft Guide,generate_interview_guide,Generated 5 questions,APPROVED,The structured interview guide meets all rubri...
5,6,Done,finalize,Guide approved. Ready for human review.,DONE,All quality gates passed.


In [ ]:
# Show final interview guide
print("\n" + "=" * 65)
print("  📋 STRUCTURED INTERVIEW GUIDE")
print("=" * 65 + "\n")

final_df = pd.DataFrame(state["guide"])
final_df


  📋 STRUCTURED INTERVIEW GUIDE



,competency,question,evidence,anchor_1,anchor_3,anchor_5
0,Communication and Relationship Building,Tell me about a time you handled a challenging...,Web search evidence identifies 'Communication ...,"Unable to describe specific actions, avoided t...",Provided a relevant example with some positive...,"Described a complex situation, used strong com..."
1,Problem Solving and Proactive Risk Management,Describe a time you identified a customer at r...,Web search evidence highlights 'Proactivity & ...,Did not identify risk early or failed to take ...,Recognized at-risk customer and took basic act...,"Anticipated risk using relevant indicators, im..."
2,Data Analysis and Metrics Orientation,How have you used customer health scores or re...,Web evidence stresses 'Data Analysis & Metrics...,"No clear use of data or metrics, acted mainly ...","Referenced metrics to inform a decision, with ...",Provided a detailed example utilizing metrics ...
3,Project Management and Organization,Explain how you managed a customer onboarding ...,JD cites onboarding and coordinating with inte...,"Failed to organize or align stakeholders, resu...",Managed the onboarding with most stakeholders ...,Clearly structured onboarding across functions...
4,Feedback and Adaptability,Tell me about a time you received critical fee...,Web and job description evidence both stress t...,Reacted defensively or made no change in behav...,Accepted feedback and took some steps to impro...,"Embraced feedback constructively, implemented ..."
